In [1]:
import pandas as pd
import numpy as np

In [4]:
import pandas as pd
import numpy as np

online_clean = pd.read_csv(
    "../data/processed/online_retail_clean.csv",
    dtype={
        "Invoice": "string",
        "StockCode": "string",
        "Description": "string",
        "Country": "string"
    }
)

online_clean["InvoiceDate"] = pd.to_datetime(
    online_clean["InvoiceDate"]
)

print(online_clean.shape)
print(online_clean.head())

(1003762, 8)
  Invoice StockCode                          Description  Quantity  \
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434    79323P                   PINK CHERRY LIGHTS        12   
2  489434    79323W                  WHITE CHERRY LIGHTS        12   
3  489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4  489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   

          InvoiceDate  Price  Customer ID         Country  
0 2009-12-01 07:45:00   6.95      13085.0  United Kingdom  
1 2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
2 2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
3 2009-12-01 07:45:00   2.10      13085.0  United Kingdom  
4 2009-12-01 07:45:00   1.25      13085.0  United Kingdom  


In [5]:
online_clean.dtypes

Invoice        string[python]
StockCode      string[python]
Description    string[python]
Quantity                int64
InvoiceDate    datetime64[ns]
Price                 float64
Customer ID           float64
Country        string[python]
dtype: object

In [6]:
online_clean["Year"] = online_clean["InvoiceDate"].dt.year
online_clean["Month"] = online_clean["InvoiceDate"].dt.month
online_clean["Day"] = online_clean["InvoiceDate"].dt.day
online_clean["DayOfWeek"] = online_clean["InvoiceDate"].dt.dayofweek
online_clean["Hour"] = online_clean["InvoiceDate"].dt.hour
online_clean["IsWeekend"] = online_clean["DayOfWeek"] >= 5

In [7]:
online_clean[
    [
        "InvoiceDate",
        "Year",
        "Month",
        "Day",
        "DayOfWeek",
        "Hour",
        "IsWeekend"
    ]
].head(10)

,InvoiceDate,Year,Month,Day,DayOfWeek,Hour,IsWeekend
0,2009-12-01 07:45:00,2009,12,1,1,7,False
1,2009-12-01 07:45:00,2009,12,1,1,7,False
2,2009-12-01 07:45:00,2009,12,1,1,7,False
3,2009-12-01 07:45:00,2009,12,1,1,7,False
4,2009-12-01 07:45:00,2009,12,1,1,7,False
5,2009-12-01 07:45:00,2009,12,1,1,7,False
6,2009-12-01 07:45:00,2009,12,1,1,7,False
7,2009-12-01 07:45:00,2009,12,1,1,7,False
8,2009-12-01 07:46:00,2009,12,1,1,7,False
9,2009-12-01 07:46:00,2009,12,1,1,7,False


In [8]:
print(online_clean.shape)
print(online_clean.isnull().sum())

(1003762, 14)
Invoice             0
StockCode           0
Description         0
Quantity            0
InvoiceDate         0
Price               0
Customer ID    226869
Country             0
Year                0
Month               0
Day                 0
DayOfWeek           0
Hour                0
IsWeekend           0
dtype: int64


In [9]:
online_clean["Revenue"] = online_clean["Quantity"] * online_clean["Price"]

In [10]:
online_clean[
    ["Invoice", "StockCode", "Quantity", "Price", "Revenue"]
].head(10)

,Invoice,StockCode,Quantity,Price,Revenue
0,489434,85048,12,6.95,83.4
1,489434,79323P,12,6.75,81.0
2,489434,79323W,12,6.75,81.0
3,489434,22041,48,2.10,100.8
4,489434,21232,24,1.25,30.0
5,489434,22064,24,1.65,39.6
6,489434,21871,24,1.25,30.0
7,489434,21523,10,5.95,59.5
8,489435,22350,12,2.55,30.6
9,489435,22349,12,3.75,45.0


In [11]:
online_clean["Revenue"].describe()

count    1.003762e+06
mean     1.959840e+01
std      2.002462e+02
min      1.000000e-03
25%      4.130000e+00
50%      1.008000e+01
75%      1.770000e+01
max      1.684696e+05
Name: Revenue, dtype: float64

In [12]:
online_clean["MonthYear"] = online_clean["InvoiceDate"].dt.to_period("M")

In [13]:
online_clean["Quarter"] = online_clean["InvoiceDate"].dt.quarter

In [14]:
online_clean[
    ["InvoiceDate", "Year", "Month", "MonthYear", "Quarter"]
].head(10)

,InvoiceDate,Year,Month,MonthYear,Quarter
0,2009-12-01 07:45:00,2009,12,2009-12,4
1,2009-12-01 07:45:00,2009,12,2009-12,4
2,2009-12-01 07:45:00,2009,12,2009-12,4
3,2009-12-01 07:45:00,2009,12,2009-12,4
4,2009-12-01 07:45:00,2009,12,2009-12,4
5,2009-12-01 07:45:00,2009,12,2009-12,4
6,2009-12-01 07:45:00,2009,12,2009-12,4
7,2009-12-01 07:45:00,2009,12,2009-12,4
8,2009-12-01 07:46:00,2009,12,2009-12,4
9,2009-12-01 07:46:00,2009,12,2009-12,4


In [15]:
print(online_clean.shape)
print(online_clean["MonthYear"].min())
print(online_clean["MonthYear"].max())
print(online_clean["Quarter"].value_counts().sort_index())

(1003762, 17)
2009-12
2011-12
Quarter
1    193610
2    204185
3    225416
4    380551
Name: count, dtype: int64


In [16]:
product_summary = online_clean.groupby(
    ["StockCode", "Description"]
).agg(
    Total_Quantity=("Quantity", "sum"),
    Total_Revenue=("Revenue", "sum"),
    Transaction_Count=("Invoice", "nunique"),
    Average_Price=("Price", "mean")
).reset_index()

In [17]:
print(product_summary.shape)
product_summary.head(10)

(5624, 6)


,StockCode,Description,Total_Quantity,Total_Revenue,Transaction_Count,Average_Price
0,10002,INFLATABLE POLITICAL GLOBE,8671,6942.26,362,0.979155
1,10002R,ROBOT PENCIL SHARPNER,4,20.57,3,5.133333
2,10080,GROOVY CACTUS INFLATABLE,315,129.29,27,0.505000
3,10109,BENDY COLOUR PENCILS,4,1.68,1,0.420000
4,10120,DOGGY RUBBER,664,142.74,73,0.240137
5,10123C,HEARTS WRAPPING TAPE,650,254.41,63,0.790635
6,10123G,ARMY CAMO WRAPPING TAPE,2251,165.56,16,0.820588
7,10124A,SPOTS ON RED BOOKCOVER TAPE,58,24.36,18,0.420000
8,10124G,ARMY CAMO BOOKCOVER TAPE,33,14.28,10,0.462000
9,10125,MINI FUNKY DESIGN TAPES,2103,1715.78,167,0.930636


In [18]:
product_summary.describe()

,Total_Quantity,Total_Revenue,Transaction_Count,Average_Price
count,5624.000000,5624.000000,5624.000000,5624.000000
mean,1989.456259,3497.889749,176.468883,6.448074
std,5102.089104,9794.682690,292.524189,147.950235
min,1.000000,0.017000,1.000000,0.001000
25%,77.000000,189.197500,17.000000,1.412341
50%,471.000000,914.855000,70.000000,2.536190
75%,1817.250000,3036.012500,211.000000,4.874492
max,106139.000000,330590.320000,5356.000000,11062.060000


In [19]:
monthly_product = online_clean.groupby(
    ["StockCode", "Description", "MonthYear"]
).agg(
    Monthly_Quantity=("Quantity", "sum"),
    Monthly_Revenue=("Revenue", "sum"),
    Monthly_Transactions=("Invoice", "nunique"),
    Average_Monthly_Price=("Price", "mean")
).reset_index()

In [20]:
print(monthly_product.shape)
monthly_product.head(10)

(68102, 7)


,StockCode,Description,MonthYear,Monthly_Quantity,Monthly_Revenue,Monthly_Transactions,Average_Monthly_Price
0,10002,INFLATABLE POLITICAL GLOBE,2009-12,215,185.30,20,0.977500
1,10002,INFLATABLE POLITICAL GLOBE,2010-01,291,248.97,17,0.945294
2,10002,INFLATABLE POLITICAL GLOBE,2010-02,257,220.07,15,0.958000
3,10002,INFLATABLE POLITICAL GLOBE,2010-03,641,493.09,17,0.911111
4,10002,INFLATABLE POLITICAL GLOBE,2010-04,1132,843.73,24,0.924167
5,10002,INFLATABLE POLITICAL GLOBE,2010-05,1465,1157.49,33,0.953824
6,10002,INFLATABLE POLITICAL GLOBE,2010-06,463,404.20,37,0.999730
7,10002,INFLATABLE POLITICAL GLOBE,2010-07,512,416.49,25,0.969615
8,10002,INFLATABLE POLITICAL GLOBE,2010-08,586,450.15,23,1.014783
9,10002,INFLATABLE POLITICAL GLOBE,2010-09,240,213.72,20,0.971500


In [21]:
monthly_product.describe()

,Monthly_Quantity,Monthly_Revenue,Monthly_Transactions,Average_Monthly_Price
count,68102.000000,68102.000000,68102.000000,68102.000000
mean,164.293295,288.862764,14.573155,4.052396
std,606.480670,1050.892471,21.813532,43.093999
min,1.000000,0.001000,1.000000,0.001000
25%,7.000000,18.670000,2.000000,1.250000
50%,35.000000,68.850000,6.000000,2.437500
75%,143.000000,250.590000,18.000000,4.300000
max,80995.000000,168469.600000,487.000000,11062.060000


In [22]:
print("Start:", monthly_product["MonthYear"].min())
print("End:", monthly_product["MonthYear"].max())

print("Unique products:", monthly_product["StockCode"].nunique())
print("Unique months:", monthly_product["MonthYear"].nunique())

Start: 2009-12
End: 2011-12
Unique products: 4912
Unique months: 25


In [23]:
product_month_counts = (
    monthly_product.groupby("StockCode")["MonthYear"]
    .nunique()
)

print(product_month_counts.describe())

count    4912.000000
mean       13.636197
std         8.179379
min         1.000000
25%         6.000000
50%        14.000000
75%        22.000000
max        25.000000
Name: MonthYear, dtype: float64


In [24]:
print(
    "Products with fewer than 25 observed months:",
    (product_month_counts < 25).sum()
)

print(
    "Products with all 25 months:",
    (product_month_counts == 25).sum()
)

Products with fewer than 25 observed months: 4236
Products with all 25 months: 676


In [25]:
product_date_range = (
    monthly_product.groupby("StockCode")["MonthYear"]
    .agg(["min", "max", "nunique"])
    .reset_index()
)

product_date_range.head(10)

,StockCode,min,max,nunique
0,10002,2009-12,2011-04,17
1,10002R,2009-12,2010-01,2
2,10080,2009-12,2011-11,11
3,10109,2009-12,2009-12,1
4,10120,2009-12,2011-12,21
5,10123C,2009-12,2011-03,14
6,10123G,2009-12,2010-11,6
7,10124A,2010-03,2011-11,12
8,10124G,2010-05,2011-11,9
9,10125,2009-12,2011-12,23


In [26]:
product_date_range["Expected_Months"] = (
    product_date_range["max"] - product_date_range["min"]
).apply(lambda x: x.n + 1)

product_date_range["Missing_Months"] = (
    product_date_range["Expected_Months"]
    - product_date_range["nunique"]
)

product_date_range.head(10)

,StockCode,min,max,nunique,Expected_Months,Missing_Months
0,10002,2009-12,2011-04,17,17,0
1,10002R,2009-12,2010-01,2,2,0
2,10080,2009-12,2011-11,11,24,13
3,10109,2009-12,2009-12,1,1,0
4,10120,2009-12,2011-12,21,25,4
5,10123C,2009-12,2011-03,14,16,2
6,10123G,2009-12,2010-11,6,12,6
7,10124A,2010-03,2011-11,12,21,9
8,10124G,2010-05,2011-11,9,19,10
9,10125,2009-12,2011-12,23,25,2


In [27]:
print(
    "Products with internal missing months:",
    (product_date_range["Missing_Months"] > 0).sum()
)

print(
    "Total internal missing months:",
    product_date_range["Missing_Months"].sum()
)

Products with internal missing months: 2265
Total internal missing months: 12443


In [28]:
duplicate_product_months = (
    monthly_product.groupby(["StockCode", "MonthYear"])
    .size()
    .reset_index(name="Count")
)

print(
    "Duplicate StockCode-Month combinations:",
    (duplicate_product_months["Count"] > 1).sum()
)

duplicate_product_months[
    duplicate_product_months["Count"] > 1
].head(20)

Duplicate StockCode-Month combinations: 1083


,StockCode,MonthYear,Count
634,16011,2010-03,2
926,16156L,2011-03,2
1030,16161U,2010-07,2
2077,17107D,2011-09,2
2127,17129F,2010-07,2
2263,18094C,2010-02,2
2512,20622,2011-06,2
2528,20652,2010-09,2
2648,20658,2010-09,2
2701,20661,2010-09,2


In [29]:
print(
    "Total StockCode-Month combinations:",
    len(duplicate_product_months)
)

Total StockCode-Month combinations: 66981


In [30]:
monthly_demand = online_clean.groupby(
    ["StockCode", "MonthYear"]
).agg(
    Monthly_Quantity=("Quantity", "sum"),
    Monthly_Revenue=("Revenue", "sum"),
    Monthly_Transactions=("Invoice", "nunique"),
    Average_Monthly_Price=("Price", "mean")
).reset_index()

In [31]:
print("Shape:", monthly_demand.shape)

print(
    "Duplicate StockCode-Month combinations:",
    monthly_demand.duplicated(
        ["StockCode", "MonthYear"]
    ).sum()
)

Shape: (66981, 6)
Duplicate StockCode-Month combinations: 0


In [32]:
monthly_demand.head(10)

,StockCode,MonthYear,Monthly_Quantity,Monthly_Revenue,Monthly_Transactions,Average_Monthly_Price
0,10002,2009-12,215,185.30,20,0.977500
1,10002,2010-01,291,248.97,17,0.945294
2,10002,2010-02,257,220.07,15,0.958000
3,10002,2010-03,641,493.09,17,0.911111
4,10002,2010-04,1132,843.73,24,0.924167
5,10002,2010-05,1465,1157.49,33,0.953824
6,10002,2010-06,463,404.20,37,0.999730
7,10002,2010-07,512,416.49,25,0.969615
8,10002,2010-08,586,450.15,23,1.014783
9,10002,2010-09,240,213.72,20,0.971500


In [33]:
print(
    "Original total quantity:",
    online_clean["Quantity"].sum()
)

print(
    "Monthly total quantity:",
    monthly_demand["Monthly_Quantity"].sum()
)

print(
    "Original total revenue:",
    online_clean["Revenue"].sum()
)

print(
    "Monthly total revenue:",
    monthly_demand["Monthly_Revenue"].sum()
)

Original total quantity: 11188702
Monthly total quantity: 11188702
Original total revenue: 19672131.948000003
Monthly total revenue: 19672131.948


In [34]:
print(
    "Unique products:",
    monthly_demand["StockCode"].nunique()
)

print(
    "Unique months:",
    monthly_demand["MonthYear"].nunique()
)

print(
    "Start:",
    monthly_demand["MonthYear"].min()
)

print(
    "End:",
    monthly_demand["MonthYear"].max()
)

Unique products: 4912
Unique months: 25
Start: 2009-12
End: 2011-12


In [35]:
product_ranges = (
    monthly_demand.groupby("StockCode")["MonthYear"]
    .agg(["min", "max"])
    .reset_index()
)

product_ranges.head()

,StockCode,min,max
0,10002,2009-12,2011-04
1,10002R,2009-12,2010-01
2,10080,2009-12,2011-11
3,10109,2009-12,2009-12
4,10120,2009-12,2011-12


In [36]:
product_ranges["MonthYear"] = product_ranges.apply(
    lambda row: pd.period_range(
        start=row["min"],
        end=row["max"],
        freq="M"
    ),
    axis=1
)

In [37]:
complete_product_months = (
    product_ranges[["StockCode", "MonthYear"]]
    .explode("MonthYear", ignore_index=True)
)

In [38]:
print("Complete product-month rows:", len(complete_product_months))
complete_product_months.head(10)

Complete product-month rows: 79424


,StockCode,MonthYear
0,10002,2009-12
1,10002,2010-01
2,10002,2010-02
3,10002,2010-03
4,10002,2010-04
5,10002,2010-05
6,10002,2010-06
7,10002,2010-07
8,10002,2010-08
9,10002,2010-09


In [39]:
monthly_demand_complete = complete_product_months.merge(
    monthly_demand,
    on=["StockCode", "MonthYear"],
    how="left"
)

In [40]:
zero_columns = [
    "Monthly_Quantity",
    "Monthly_Revenue",
    "Monthly_Transactions"
]

monthly_demand_complete[zero_columns] = (
    monthly_demand_complete[zero_columns].fillna(0)
)

In [41]:
print("Original monthly rows:", len(monthly_demand))
print("Complete monthly rows:", len(monthly_demand_complete))

print(
    "Missing Quantity:",
    monthly_demand_complete["Monthly_Quantity"].isna().sum()
)

print(
    "Missing Revenue:",
    monthly_demand_complete["Monthly_Revenue"].isna().sum()
)

print(
    "Missing Transactions:",
    monthly_demand_complete["Monthly_Transactions"].isna().sum()
)

Original monthly rows: 66981
Complete monthly rows: 79424
Missing Quantity: 0
Missing Revenue: 0
Missing Transactions: 0


In [42]:
print(
    "Total quantity:",
    monthly_demand_complete["Monthly_Quantity"].sum()
)

print(
    "Total revenue:",
    monthly_demand_complete["Monthly_Revenue"].sum()
)

Total quantity: 11188702.0
Total revenue: 19672131.948


In [43]:
print(
    "Zero-demand product-months:",
    (
        monthly_demand_complete["Monthly_Quantity"] == 0
    ).sum()
)

Zero-demand product-months: 12443


In [44]:
monthly_demand_complete = monthly_demand_complete.sort_values(
    ["StockCode", "MonthYear"]
).reset_index(drop=True)

In [45]:
monthly_demand_complete["Lag_1"] = (
    monthly_demand_complete
    .groupby("StockCode")["Monthly_Quantity"]
    .shift(1)
)

monthly_demand_complete["Lag_2"] = (
    monthly_demand_complete
    .groupby("StockCode")["Monthly_Quantity"]
    .shift(2)
)

monthly_demand_complete["Lag_3"] = (
    monthly_demand_complete
    .groupby("StockCode")["Monthly_Quantity"]
    .shift(3)
)

In [46]:
monthly_demand_complete[
    [
        "StockCode",
        "MonthYear",
        "Monthly_Quantity",
        "Lag_1",
        "Lag_2",
        "Lag_3"
    ]
].head(15)

,StockCode,MonthYear,Monthly_Quantity,Lag_1,Lag_2,Lag_3
0,10002,2009-12,215.0,NaN,NaN,NaN
1,10002,2010-01,291.0,215.0,NaN,NaN
2,10002,2010-02,257.0,291.0,215.0,NaN
3,10002,2010-03,641.0,257.0,291.0,215.0
4,10002,2010-04,1132.0,641.0,257.0,291.0
5,10002,2010-05,1465.0,1132.0,641.0,257.0
6,10002,2010-06,463.0,1465.0,1132.0,641.0
7,10002,2010-07,512.0,463.0,1465.0,1132.0
8,10002,2010-08,586.0,512.0,463.0,1465.0
9,10002,2010-09,240.0,586.0,512.0,463.0


In [47]:
monthly_demand_complete["Rolling_3_Month"] = (
    monthly_demand_complete
    .groupby("StockCode")["Monthly_Quantity"]
    .transform(
        lambda x: x.shift(1).rolling(3).mean()
    )
)

monthly_demand_complete["Rolling_6_Month"] = (
    monthly_demand_complete
    .groupby("StockCode")["Monthly_Quantity"]
    .transform(
        lambda x: x.shift(1).rolling(6).mean()
    )
)

In [48]:
monthly_demand_complete[
    [
        "StockCode",
        "MonthYear",
        "Monthly_Quantity",
        "Lag_1",
        "Lag_2",
        "Lag_3",
        "Rolling_3_Month",
        "Rolling_6_Month"
    ]
].head(15)

,StockCode,MonthYear,Monthly_Quantity,Lag_1,Lag_2,Lag_3,Rolling_3_Month,Rolling_6_Month
0,10002,2009-12,215.0,NaN,NaN,NaN,NaN,NaN
1,10002,2010-01,291.0,215.0,NaN,NaN,NaN,NaN
2,10002,2010-02,257.0,291.0,215.0,NaN,NaN,NaN
3,10002,2010-03,641.0,257.0,291.0,215.0,254.333333,NaN
4,10002,2010-04,1132.0,641.0,257.0,291.0,396.333333,NaN
5,10002,2010-05,1465.0,1132.0,641.0,257.0,676.666667,NaN
6,10002,2010-06,463.0,1465.0,1132.0,641.0,1079.333333,666.833333
7,10002,2010-07,512.0,463.0,1465.0,1132.0,1020.000000,708.166667
8,10002,2010-08,586.0,512.0,463.0,1465.0,813.333333,745.000000
9,10002,2010-09,240.0,586.0,512.0,463.0,520.333333,799.833333


In [49]:
print(monthly_demand_complete.shape)

(79424, 11)


In [50]:
monthly_demand_complete["Lag_1_Price"] = (
    monthly_demand_complete
    .groupby("StockCode")["Average_Monthly_Price"]
    .shift(1)
)

In [51]:
monthly_demand_complete["Price_Change_Pct"] = (
    (
        monthly_demand_complete["Average_Monthly_Price"]
        - monthly_demand_complete["Lag_1_Price"]
    )
    / monthly_demand_complete["Lag_1_Price"]
) * 100

In [52]:
monthly_demand_complete[
    [
        "StockCode",
        "MonthYear",
        "Average_Monthly_Price",
        "Lag_1_Price",
        "Price_Change_Pct"
    ]
].head(20)

,StockCode,MonthYear,Average_Monthly_Price,Lag_1_Price,Price_Change_Pct
0,10002,2009-12,0.977500,NaN,NaN
1,10002,2010-01,0.945294,0.977500,-3.294719
2,10002,2010-02,0.958000,0.945294,1.344119
3,10002,2010-03,0.911111,0.958000,-4.894456
4,10002,2010-04,0.924167,0.911111,1.432927
5,10002,2010-05,0.953824,0.924167,3.209038
6,10002,2010-06,0.999730,0.953824,4.812861
7,10002,2010-07,0.969615,0.999730,-3.012249
8,10002,2010-08,1.014783,0.969615,4.658262
9,10002,2010-09,0.971500,1.014783,-4.265210


In [53]:
print(monthly_demand_complete.shape)

(79424, 13)


In [54]:
price_columns = [
    "Average_Monthly_Price",
    "Lag_1_Price",
    "Price_Change_Pct"
]

print(monthly_demand_complete[price_columns].isna().sum())

Average_Monthly_Price    12443
Lag_1_Price              17355
Price_Change_Pct         23071
dtype: int64


In [55]:
monthly_demand_complete["Price_Change_Pct"].describe()

count    56353.000000
mean         3.151157
std         41.867085
min        -98.370008
25%         -6.860594
50%          0.000000
75%          7.872219
max       3164.705882
Name: Price_Change_Pct, dtype: float64

In [56]:
monthly_demand_complete[
    monthly_demand_complete["Price_Change_Pct"].abs() > 50
][
    [
        "StockCode",
        "MonthYear",
        "Average_Monthly_Price",
        "Lag_1_Price",
        "Price_Change_Pct"
    ]
].head(20)

,StockCode,MonthYear,Average_Monthly_Price,Lag_1_Price,Price_Change_Pct
46,10120,2010-02,0.430000,0.283333,51.764706
47,10120,2010-03,0.210000,0.430000,-51.162791
72,10123C,2010-03,1.028000,0.650000,58.153846
93,10123G,2010-08,0.060000,0.650000,-90.769231
95,10123G,2010-10,0.650000,0.060000,983.333333
96,10123G,2010-11,1.028750,0.650000,58.269231
138,10125,2010-01,1.660000,0.956250,73.594771
210,10135,2011-02,1.669167,1.061667,57.221350
246,11001,2011-08,1.816667,1.021111,77.910773
280,15034,2010-04,0.475000,0.274000,73.357664


In [57]:
monthly_demand_complete["Price_Ratio"] = (
    monthly_demand_complete["Average_Monthly_Price"]
    / monthly_demand_complete["Lag_1_Price"]
)

In [58]:
monthly_demand_complete["Log_Price_Ratio"] = np.log(
    monthly_demand_complete["Price_Ratio"]
)

In [59]:
monthly_demand_complete[
    [
        "StockCode",
        "MonthYear",
        "Average_Monthly_Price",
        "Lag_1_Price",
        "Price_Change_Pct",
        "Price_Ratio",
        "Log_Price_Ratio"
    ]
].head(20)

,StockCode,MonthYear,Average_Monthly_Price,Lag_1_Price,Price_Change_Pct,Price_Ratio,Log_Price_Ratio
0,10002,2009-12,0.977500,NaN,NaN,NaN,NaN
1,10002,2010-01,0.945294,0.977500,-3.294719,0.967053,-0.033502
2,10002,2010-02,0.958000,0.945294,1.344119,1.013441,0.013352
3,10002,2010-03,0.911111,0.958000,-4.894456,0.951055,-0.050183
4,10002,2010-04,0.924167,0.911111,1.432927,1.014329,0.014228
5,10002,2010-05,0.953824,0.924167,3.209038,1.032090,0.031586
6,10002,2010-06,0.999730,0.953824,4.812861,1.048129,0.047006
7,10002,2010-07,0.969615,0.999730,-3.012249,0.969878,-0.030585
8,10002,2010-08,1.014783,0.969615,4.658262,1.046583,0.045530
9,10002,2010-09,0.971500,1.014783,-4.265210,0.957348,-0.043588


In [60]:
print(monthly_demand_complete.shape)

(79424, 15)


In [61]:
monthly_demand_complete.to_csv(
    "../data/processed/monthly_demand_features.csv",
    index=False
)

print("Feature-engineered dataset saved successfully.")
print(monthly_demand_complete.shape)

Feature-engineered dataset saved successfully.
(79424, 15)
